# Lean-RGC v52 Colab Setup and Bivariate Kernel-Cache Experiment

This notebook is for Google Colab. It clones the public repository, installs Python dependencies, installs Lean through elan, builds the Lean template project, runs kernel RPC smoke checks, and starts the v52 bivariate contextual quotient experiment.

GPU is not required for these Lean/kernel experiments. Use a CPU runtime first. GPU only matters for later ML model training over collected artifacts.

## 0. Runtime Notes

- Recommended Colab runtime: CPU, high-RAM optional.
- Keep the tab open while long Lean checks run.
- If a cell is interrupted, restart from the last completed setup cell and use a new `RUN` directory.
- The full bivariate audit can be heavy. Start with the smoke experiment before increasing budgets.

In [ ]:
import os

REPO_URL = "https://github.com/abhorrence-of-Gods/lean-rgc-automation-stack-v51.git"
REPO_DIR = "/content/lean-rgc-automation-stack-v51"
os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = REPO_DIR
print(REPO_URL)
print(REPO_DIR)

## 1. Clone Repository

In [ ]:
!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git log --oneline --decorate -5

## 2. Install Python Package

Colab already has Python. Install this repo in editable mode and add pytest for smoke tests.

In [ ]:
%cd "$REPO_DIR"
!python -m pip install --upgrade pip
!python -m pip install -e . pytest
!lean-rgc --help | head -40

## 3. Install Lean / elan

This installs elan into `/root/.elan` and exposes `lean`/`lake` to later cells.

In [ ]:
import os
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
!test -x "$HOME/.elan/bin/elan" || curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y
!source "$HOME/.elan/env" && lean --version
!source "$HOME/.elan/env" && lake --version

## 4. Build Lean Template Project

The zip/repo intentionally does not include `.lake` build cache. Colab rebuilds it here.

In [ ]:
%cd "$REPO_DIR/lean_project_template"
!source "$HOME/.elan/env" && lake update
!source "$HOME/.elan/env" && lake build
%cd "$REPO_DIR"

## 5. Python Smoke Tests

These tests validate the bivariate row/column design and the v52 kernel context cache without running a large experiment.

In [ ]:
%cd "$REPO_DIR"
!python -m pytest -q tests/test_v51_bivariate_contextual_quotient.py tests/test_v52_kernel_context_cache.py tests/test_v49_kernel_rpc_worker.py

## 6. Native Worker / Kernel RPC Smoke

This verifies that the packaged Lean-side JSONL worker starts and accepts kernel RPC commands.

In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"
source "$HOME/.elan/env"
python -m lean_rgc.native_worker --lean-cmd "lake env lean" --exec-mode kernel_rpc --workdir lean_project_template --print-command
printf '%s\n' '{"id":"load","cmd":"load_project","imports":["Lean"]}' '{"id":"stop","cmd":"shutdown"}' \
  | python -m lean_rgc.native_worker --lean-cmd "lake env lean" --exec-mode kernel_rpc --workdir lean_project_template

## 7. Baseline Source-Check Smoke

This checks the older source-check path. It is slower than kernel RPC but useful as a sanity baseline.

In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"
source "$HOME/.elan/env"
RUN="runs/colab_v52_source_smoke_$(date +%Y%m%d_%H%M%S)"
mkdir -p "$(dirname "$RUN")"
lean-rgc pipeline \
  --tasks examples/core_theorems.jsonl \
  --actions examples/core_tactics.jsonl \
  --out "$RUN" \
  --audit-mode server \
  --server-backend native \
  --native-exec-mode source_check \
  --server-no-fallback \
  --lean-cmd "lake env lean" \
  --workdir lean_project_template \
  --import-mode core \
  --max-actions 40 \
  2>&1 | tee "$RUN.log"
echo "RUN=$RUN"

## 8. v52 Bivariate Kernel-Cache Smoke

This is the first important experiment. It uses `kernel_rpc`, decomposes contextual actions into `B`, `u`, `A`, and caches kernel states for repeated contexts.

Expected key signs:

- `kernel_context_cache = true`
- `source_check_calls = 0`
- `context_cache_hits > 0`
- `baseline_missing = 0`
- `row_degenerate = false`
- `column_degenerate = false`

In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"
source "$HOME/.elan/env"
RUN="runs/colab_v52_bivariate_kernel_cache_smoke_$(date +%Y%m%d_%H%M%S)"
mkdir -p "$(dirname "$RUN")"
lean-rgc pipeline \
  --tasks examples/core_theorems.jsonl \
  --actions examples/core_tactics.jsonl \
  --out "$RUN" \
  --audit-mode server \
  --server-backend native \
  --native-exec-mode kernel_rpc \
  --server-no-fallback \
  --lean-cmd "lake env lean" \
  --workdir lean_project_template \
  --import-mode core \
  --max-actions 80 \
  --premise-contextual-bivariate \
  --premise-contextual-quotient \
  --premise-contextual-max-premises 8 \
  --premise-contextual-max-left 4 \
  --premise-contextual-max-right 8 \
  --bivariate-audit-budget 96 \
  --premise-contextual-baseline-required \
  --skip-vacuous-premise-quotient \
  --repair-face-ledger \
  2>&1 | tee "$RUN.log"
echo "RUN=$RUN"

## 9. Inspect Latest v52 Results

In [ ]:
import json
from pathlib import Path

%cd "$REPO_DIR"
runs = sorted(Path("runs").glob("colab_v52_bivariate_kernel_cache_smoke_*"))
if not runs:
    raise SystemExit("No v52 kernel-cache smoke run found")
run = runs[-1]
print("RUN:", run)

for rel in [
    "audit/server_summary.json",
    "premise_contextual_audit/server_summary.json",
    "premise_contextual/premise_use_row_report.json",
    "premise_contextual/bivariate_contextual_candidate_report.json",
    "premise_contextual/bivariate_contextual_schedule_report.json",
    "premise_contextual/premise_contextual_fingerprint_report.json",
    "premise_contextual/premise_contextual_quotient_report.json",
    "premise_contextual_audit/contextual_plan_audit_report.json",
]:
    p = run / rel
    print("\n==", rel, "==")
    if not p.exists():
        print("missing")
        continue
    d = json.loads(p.read_text())
    keep = {k: d.get(k) for k in [
        "n", "n_responses", "n_failures", "statuses", "success_rate",
        "kernel_context_cache", "context_cache_entries", "context_cache_hits",
        "context_cache_misses", "source_check_calls", "n_rows", "n_actions",
        "n_premise_use_rows", "n_context_pairs", "n_selected",
        "n_selected_probes", "n_selected_baselines", "baseline_missing",
        "n_fingerprints", "n_unique_premise_use_ids", "n_unique_context_pairs",
        "row_degenerate", "column_degenerate", "quotient_status", "reason",
        "n_classes", "n_faces"
    ]}
    print(json.dumps({k:v for k,v in keep.items() if v is not None}, indent=2))

## 10. Medium Bivariate Run

Run this only after the smoke experiment is healthy. It increases row count and audit budget.

In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"
source "$HOME/.elan/env"
RUN="runs/colab_v52_bivariate_kernel_cache_medium_$(date +%Y%m%d_%H%M%S)"
mkdir -p "$(dirname "$RUN")"
lean-rgc pipeline \
  --tasks examples/core_theorems.jsonl \
  --actions examples/core_tactics.jsonl \
  --out "$RUN" \
  --audit-mode server \
  --server-backend native \
  --native-exec-mode kernel_rpc \
  --server-no-fallback \
  --lean-cmd "lake env lean" \
  --workdir lean_project_template \
  --import-mode core \
  --max-actions 200 \
  --premise-contextual-bivariate \
  --premise-contextual-quotient \
  --premise-contextual-max-premises 16 \
  --premise-contextual-max-left 4 \
  --premise-contextual-max-right 8 \
  --bivariate-audit-budget 192 \
  --premise-contextual-baseline-required \
  --skip-vacuous-premise-quotient \
  --repair-face-ledger \
  --premise-quotient-retrieve \
  2>&1 | tee "$RUN.log"
echo "RUN=$RUN"

## 11. Optional Full-ish Run

This can take much longer. Run it only after smoke/medium succeed.

In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"
source "$HOME/.elan/env"
RUN="runs/colab_v52_bivariate_kernel_cache_full_$(date +%Y%m%d_%H%M%S)"
mkdir -p "$(dirname "$RUN")"
lean-rgc pipeline \
  --tasks examples/core_theorems.jsonl \
  --actions examples/core_tactics.jsonl \
  --out "$RUN" \
  --audit-mode server \
  --server-backend native \
  --native-exec-mode kernel_rpc \
  --server-no-fallback \
  --lean-cmd "lake env lean" \
  --workdir lean_project_template \
  --import-mode core \
  --max-actions 500 \
  --premise-contextual-bivariate \
  --premise-contextual-quotient \
  --premise-contextual-max-premises 32 \
  --premise-contextual-max-left 4 \
  --premise-contextual-max-right 8 \
  --bivariate-audit-budget 512 \
  --premise-contextual-baseline-required \
  --skip-vacuous-premise-quotient \
  --repair-face-ledger \
  --premise-quotient-retrieve \
  2>&1 | tee "$RUN.log"
echo "RUN=$RUN"

## 12. Package and Download Results

In [ ]:
%cd "$REPO_DIR"
!zip -r /content/lean_rgc_colab_results.zip runs -x '**/.lake/**' '**/__pycache__/**' '*.pyc'
from google.colab import files
files.download('/content/lean_rgc_colab_results.zip')